In [3]:
import os
import re
import sys
import time
import sqlite3
import statistics

from collections import defaultdict
from datetime import datetime
from typing import Any
from zoneinfo import ZoneInfo


# ==================================================
# 初期設定
# ==================================================

start_time = time.time()


# ==================================================
# importパス設定
# ==================================================

# Jupyter・通常スクリプトの両方に対応
try:
    base_dir = os.path.dirname(
        os.path.abspath(__file__)
    )
except NameError:
    base_dir = os.getcwd()


# 現在位置:
# soubanavi/scripts/database/
#
# プロジェクトルート:
# soubanavi/
project_root = os.path.abspath(
    os.path.join(
        base_dir,
        "..",
        "..",
    )
)


if project_root not in sys.path:
    sys.path.insert(
        0,
        project_root,
    )


from utils.config import (
    DB_PATH,
)


# ==================================================
# テーブル設定
# ==================================================

# 集計元テーブル
SOURCE_TABLE = "result_table"

# 集計先テーブル
HISTORY_TABLE = "price_history"


# 日本時間
JAPAN_TIMEZONE = ZoneInfo(
    "Asia/Tokyo"
)


# ==================================================
# 集計設定
# ==================================================

# True:
# 今日を含めて過去分を再集計する
#
# False:
# 昨日以前だけを再集計する
#
# 今日分はupdate_price_history.pyで保存しているため、
# 通常はFalseを推奨
INCLUDE_TODAY = False


# 指定した日付以降だけ再集計したい場合に設定
#
# 例:
# START_DATE = "2026-07-01"
#
# 全期間を処理する場合:
START_DATE = None


# 指定した日付以前だけ再集計したい場合に設定
#
# 通常はNoneでよい
END_DATE = None


# 既存のprice_historyを更新するか
#
# True:
# 同じ機種ID・日付があれば再計算結果で更新
#
# False:
# 既存データは変更しない
UPDATE_EXISTING = True


# ==================================================
# 一般補助関数
# ==================================================

def validate_identifier(
    identifier: str,
) -> str:
    """
    SQL識別子（テーブル名など）が
    英数字とアンダースコアのみか確認する。
    """
    if not re.fullmatch(
        r"[A-Za-z_][A-Za-z0-9_]*",
        identifier,
    ):
        raise ValueError(
            f"不正なSQL識別子です: {identifier}"
        )

    return identifier

def validate_database(
    connection: sqlite3.Connection,
) -> None:
    """
    使用するテーブルとカラムを確認する。
    """
    check_table_exists(
        connection,
        SOURCE_TABLE,
    )

    check_table_exists(
        connection,
        HISTORY_TABLE,
    )

    check_required_columns(
        connection,
        SOURCE_TABLE,
        {
            "id",
            "shop_name",
            "machine_name",
            "master_machine_id",
            "master_machine_name",
            "master_machine_maker",
            "price",
            "product_url",
            "created_at",
        },
    )

    check_required_columns(
        connection,
        HISTORY_TABLE,
        {
            "id",
            "master_machine_id",
            "record_date",
            "master_machine_name",
            "master_machine_maker",
            "min_price",
            "avg_price",
            "median_price",
            "max_price",
            "latest_price",
            "price_count",
            "shop_count",
            "lowest_shop_name",
            "lowest_product_url",
            "created_at",
            "updated_at",
        },
    )


def row_to_dict(
    row: sqlite3.Row,
) -> dict[str, Any]:
    """
    sqlite3.Rowを通常のdictへ変換する。
    """
    return {
        key: row[key]
        for key in row.keys()
    }


def normalize_text(
    value: Any,
) -> str:
    """
    Noneなどを安全に文字列へ変換する。
    """
    if value is None:
        return ""

    return str(value).strip()


def normalize_price(
    value: Any,
) -> int | None:
    """
    価格を正の整数へ変換する。

    変換できない値、0円以下はNoneを返す。
    """
    if value is None:
        return None

    try:
        value_text = str(value).replace(
            ",",
            "",
        ).strip()

        if not value_text:
            return None

        price = int(
            float(value_text)
        )

    except (TypeError, ValueError):
        return None

    if price <= 0:
        return None

    return price


def normalize_record_date(
    value: Any,
) -> str | None:
    """
    created_atなどの日時から
    YYYY-MM-DD形式の日付を取得する。
    """
    if value is None:
        return None

    value_text = str(value).strip()

    if not value_text:
        return None

    # SQLiteやISO形式であれば、
    # 先頭10文字がYYYY-MM-DDになる
    date_text = value_text[:10]

    try:
        parsed_date = datetime.strptime(
            date_text,
            "%Y-%m-%d",
        )

    except ValueError:
        datetime_formats = [
            "%Y/%m/%d",
            "%Y-%m-%d %H:%M:%S",
            "%Y-%m-%dT%H:%M:%S",
            "%Y/%m/%d %H:%M:%S",
        ]

        parsed_date = None

        for datetime_format in datetime_formats:
            try:
                parsed_date = datetime.strptime(
                    value_text,
                    datetime_format,
                )
                break
            except ValueError:
                continue

        if parsed_date is None:
            return None

    return parsed_date.strftime(
        "%Y-%m-%d"
    )


def calculate_median(
    prices: list[int],
) -> int | None:
    """
    価格リストの中央値を整数で返す。
    """
    if not prices:
        return None

    return int(
        statistics.median(prices)
    )


# ==================================================
# DB構造確認
# ==================================================

def check_table_exists(
    connection: sqlite3.Connection,
    target_table: str,
) -> None:
    """
    指定したテーブルが存在するか確認する。
    """
    row = connection.execute(
        """
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
          AND name = ?
        """,
        (
            target_table,
        ),
    ).fetchone()

    if row is None:
        raise RuntimeError(
            "テーブルが存在しません: "
            f"{target_table}"
        )


def get_table_columns(
    connection: sqlite3.Connection,
    target_table: str,
) -> set[str]:
    """
    指定したテーブルのカラム一覧を取得する。
    """
    safe_table_name = validate_identifier(
        target_table
    )

    rows = connection.execute(
        f"PRAGMA table_info({safe_table_name})"
    ).fetchall()

    return {
        row[1]
        for row in rows
    }


def check_required_columns(
    connection: sqlite3.Connection,
    target_table: str,
    required_columns: set[str],
) -> None:
    """
    必要なカラムが存在するか確認する。
    """
    existing_columns = get_table_columns(
        connection,
        target_table,
    )

    missing_columns = (
        required_columns
        - existing_columns
    )

    if missing_columns:
        raise RuntimeError(
            f"{target_table}に必要なカラムがありません: "
            + ", ".join(
                sorted(missing_columns)
            )
        )




# ==================================================
# 対象期間
# ==================================================

def get_target_end_date() -> str:
    """
    集計対象の終了日を決定する。
    """
    today = datetime.now(
        JAPAN_TIMEZONE
    ).date()

    if END_DATE:
        return END_DATE

    if INCLUDE_TODAY:
        return today.isoformat()

    # 今日を含めない場合、
    # SQL側でrecord_date < 今日と判定するため
    # ここでは今日の日付を返す
    return today.isoformat()


# ==================================================
# 商品データ取得
# ==================================================

def get_historical_products(
    connection: sqlite3.Connection,
) -> list[dict[str, Any]]:
    """
    result_tableから日別集計に使用する
    商品データを取得する。
    """
    safe_product_table = validate_identifier(
        SOURCE_TABLE
    )

    conditions = [
        "master_machine_id IS NOT NULL",
        """
        TRIM(
            CAST(master_machine_id AS TEXT)
        ) != ''
        """,
        "master_machine_name IS NOT NULL",
        "TRIM(master_machine_name) != ''",
        "price IS NOT NULL",
        "price > 0",
        "created_at IS NOT NULL",
        "TRIM(CAST(created_at AS TEXT)) != ''",
    ]

    parameters: list[Any] = []

    if START_DATE:
        conditions.append(
            "DATE(created_at) >= DATE(?)"
        )
        parameters.append(
            START_DATE
        )

    target_end_date = get_target_end_date()

    if END_DATE:
        conditions.append(
            "DATE(created_at) <= DATE(?)"
        )
        parameters.append(
            target_end_date
        )

    elif INCLUDE_TODAY:
        conditions.append(
            "DATE(created_at) <= DATE(?)"
        )
        parameters.append(
            target_end_date
        )

    else:
        conditions.append(
            "DATE(created_at) < DATE(?)"
        )
        parameters.append(
            target_end_date
        )

    where_clause = "\nAND ".join(
        conditions
    )

    sql = f"""
        SELECT
            id,
            shop_name,
            machine_name,
            master_machine_id,
            master_machine_name,
            master_machine_maker,
            price,
            product_url,
            created_at

        FROM {safe_product_table}

        WHERE {where_clause}

        ORDER BY
            master_machine_id ASC,
            DATE(created_at) ASC,
            created_at ASC,
            id ASC
    """

    rows = connection.execute(
        sql,
        tuple(parameters),
    ).fetchall()

    return [
        row_to_dict(row)
        for row in rows
    ]


# ==================================================
# 日別集計
# ==================================================

def aggregate_price_history(
    products: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    """
    商品データを
    機種ID・日付単位で集計する。
    """
    grouped_products: dict[
        tuple[str, str],
        list[dict[str, Any]],
    ] = defaultdict(list)

    skipped_count = 0

    for product in products:
        master_machine_id = normalize_text(
            product.get("master_machine_id")
        )

        record_date = normalize_record_date(
            product.get("created_at")
        )

        price = normalize_price(
            product.get("price")
        )

        if (
            not master_machine_id
            or not record_date
            or price is None
        ):
            skipped_count += 1
            continue

        product["normalized_price"] = price

        group_key = (
            master_machine_id,
            record_date,
        )

        grouped_products[
            group_key
        ].append(
            product
        )

    history_rows: list[
        dict[str, Any]
    ] = []

    for (
        master_machine_id,
        record_date,
    ), group in grouped_products.items():

        # 日付内の商品を取得日時・ID順に並べる
        sorted_group = sorted(
            group,
            key=lambda item: (
                normalize_text(
                    item.get("created_at")
                ),
                int(item.get("id") or 0),
            ),
        )

        prices = [
            int(item["normalized_price"])
            for item in sorted_group
        ]

        if not prices:
            continue

        # 最安商品
        lowest_product = min(
            sorted_group,
            key=lambda item: (
                int(
                    item["normalized_price"]
                ),
                normalize_text(
                    item.get("created_at")
                ),
                int(item.get("id") or 0),
            ),
        )

        # その日の最後に取得した商品の価格
        latest_product = sorted_group[-1]

        # 名称・メーカーは、
        # その日の最後の商品情報を優先する
        master_machine_name = normalize_text(
            latest_product.get(
                "master_machine_name"
            )
        )

        master_machine_maker = normalize_text(
            latest_product.get(
                "master_machine_maker"
            )
        )

        shop_names = {
            normalize_text(
                item.get("shop_name")
            )
            for item in sorted_group
            if normalize_text(
                item.get("shop_name")
            )
        }

        avg_price = int(
            round(
                sum(prices)
                / len(prices)
            )
        )

        history_rows.append(
            {
                "master_machine_id":
                    master_machine_id,

                "record_date":
                    record_date,

                "master_machine_name":
                    master_machine_name,

                "master_machine_maker":
                    master_machine_maker,

                "min_price":
                    min(prices),

                "avg_price":
                    avg_price,

                "median_price":
                    calculate_median(prices),

                "max_price":
                    max(prices),

                "latest_price":
                    int(
                        latest_product[
                            "normalized_price"
                        ]
                    ),

                "price_count":
                    len(prices),

                "shop_count":
                    len(shop_names),

                "lowest_shop_name":
                    normalize_text(
                        lowest_product.get(
                            "shop_name"
                        )
                    ),

                "lowest_product_url":
                    normalize_text(
                        lowest_product.get(
                            "product_url"
                        )
                    ),
            }
        )

    history_rows.sort(
        key=lambda item: (
            item["record_date"],
            item["master_machine_id"],
        )
    )

    print(
        "集計対象外商品数: "
        f"{skipped_count:,}件"
    )

    return history_rows


# ==================================================
# 既存データ確認
# ==================================================

def get_existing_history_keys(
    connection: sqlite3.Connection,
) -> set[tuple[str, str]]:
    """
    price_historyに保存済みの
    機種ID・日付を取得する。
    """
    safe_history_table = validate_identifier(
        HISTORY_TABLE
    )

    rows = connection.execute(
        f"""
        SELECT
            master_machine_id,
            record_date

        FROM {safe_history_table}
        """
    ).fetchall()

    return {
        (
            normalize_text(
                row["master_machine_id"]
            ),
            normalize_text(
                row["record_date"]
            ),
        )
        for row in rows
    }


# ==================================================
# price_history保存
# ==================================================

def save_price_history(
    connection: sqlite3.Connection,
    history_rows: list[dict[str, Any]],
) -> tuple[int, int, int]:
    """
    集計結果をprice_historyへ保存する。

    戻り値:
    新規件数、更新件数、スキップ件数
    """
    safe_history_table = validate_identifier(
        HISTORY_TABLE
    )

    existing_keys = get_existing_history_keys(
        connection
    )

    inserted_count = 0
    updated_count = 0
    skipped_count = 0

    now_text = datetime.now(
        JAPAN_TIMEZONE
    ).strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    if UPDATE_EXISTING:
        sql = f"""
            INSERT INTO {safe_history_table} (
                master_machine_id,
                record_date,
                master_machine_name,
                master_machine_maker,
                min_price,
                avg_price,
                median_price,
                max_price,
                latest_price,
                price_count,
                shop_count,
                lowest_shop_name,
                lowest_product_url,
                created_at,
                updated_at
            )
            VALUES (
                ?,
                ?,
                ?,
                ?,
                ?,
                ?,
                ?,
                ?,
                ?,
                ?,
                ?,
                ?,
                ?,
                ?,
                ?
            )

            ON CONFLICT(
                master_machine_id,
                record_date
            )
            DO UPDATE SET
                master_machine_name =
                    excluded.master_machine_name,

                master_machine_maker =
                    excluded.master_machine_maker,

                min_price =
                    excluded.min_price,

                avg_price =
                    excluded.avg_price,

                median_price =
                    excluded.median_price,

                max_price =
                    excluded.max_price,

                latest_price =
                    excluded.latest_price,

                price_count =
                    excluded.price_count,

                shop_count =
                    excluded.shop_count,

                lowest_shop_name =
                    excluded.lowest_shop_name,

                lowest_product_url =
                    excluded.lowest_product_url,

                updated_at =
                    excluded.updated_at
        """

    else:
        sql = f"""
            INSERT OR IGNORE INTO {safe_history_table} (
                master_machine_id,
                record_date,
                master_machine_name,
                master_machine_maker,
                min_price,
                avg_price,
                median_price,
                max_price,
                latest_price,
                price_count,
                shop_count,
                lowest_shop_name,
                lowest_product_url,
                created_at,
                updated_at
            )
            VALUES (
                ?,
                ?,
                ?,
                ?,
                ?,
                ?,
                ?,
                ?,
                ?,
                ?,
                ?,
                ?,
                ?,
                ?,
                ?
            )
        """

    for history in history_rows:
        history_key = (
            normalize_text(
                history["master_machine_id"]
            ),
            normalize_text(
                history["record_date"]
            ),
        )

        already_exists = (
            history_key in existing_keys
        )

        if (
            already_exists
            and not UPDATE_EXISTING
        ):
            skipped_count += 1
            continue

        connection.execute(
            sql,
            (
                history["master_machine_id"],
                history["record_date"],
                history["master_machine_name"],
                history["master_machine_maker"],
                history["min_price"],
                history["avg_price"],
                history["median_price"],
                history["max_price"],
                history["latest_price"],
                history["price_count"],
                history["shop_count"],
                history["lowest_shop_name"],
                history["lowest_product_url"],
                now_text,
                now_text,
            ),
        )

        if already_exists:
            updated_count += 1
        else:
            inserted_count += 1

            existing_keys.add(
                history_key
            )

    return (
        inserted_count,
        updated_count,
        skipped_count,
    )


# ==================================================
# 保存結果確認
# ==================================================

def count_history_rows(
    connection: sqlite3.Connection,
) -> int:
    """
    price_history全体の件数を取得する。
    """
    safe_history_table = validate_identifier(
        HISTORY_TABLE
    )

    row = connection.execute(
        f"""
        SELECT COUNT(*) AS count
        FROM {safe_history_table}
        """
    ).fetchone()

    return int(
        row["count"]
    )


def get_history_date_range(
    connection: sqlite3.Connection,
) -> tuple[str | None, str | None]:
    """
    price_historyの最古日・最新日を取得する。
    """
    safe_history_table = validate_identifier(
        HISTORY_TABLE
    )

    row = connection.execute(
        f"""
        SELECT
            MIN(record_date) AS min_date,
            MAX(record_date) AS max_date

        FROM {safe_history_table}
        """
    ).fetchone()

    return (
        row["min_date"],
        row["max_date"],
    )


# ==================================================
# メイン処理
# ==================================================

def backfill_price_history() -> None:
    """
    result_tableの過去商品データを日付別に集計し、
    price_historyへ保存する。
    """
    if not DB_PATH.is_file():
        raise FileNotFoundError(
            "SQLiteデータベースが見つかりません: "
            f"{DB_PATH}"
        )
    
    connection = sqlite3.connect(
        DB_PATH,
        timeout=60,
    )

    connection.row_factory = sqlite3.Row

    try:
        validate_database(
            connection
        )

        print("=" * 70)
        print("過去価格履歴の再集計を開始します。")
        print(
            "データベース: "
            f"{DB_PATH}"
        )
        
        print(
            "商品テーブル: "
            f"{SOURCE_TABLE}"
        )
        
        print(
            "履歴テーブル: "
            f"{HISTORY_TABLE}"
        )
        print(
            "開始日: "
            f"{START_DATE or '指定なし'}"
        )
        print(
            "終了日: "
            f"{END_DATE or '自動'}"
        )
        print(
            "今日を含める: "
            f"{INCLUDE_TODAY}"
        )
        print(
            "既存履歴を更新: "
            f"{UPDATE_EXISTING}"
        )
        print("=" * 70)

        products = get_historical_products(
            connection
        )

        print(
            "取得商品数: "
            f"{len(products):,}件"
        )

        history_rows = aggregate_price_history(
            products
        )

        machine_ids = {
            row["master_machine_id"]
            for row in history_rows
        }

        record_dates = {
            row["record_date"]
            for row in history_rows
        }

        print(
            "集計履歴数: "
            f"{len(history_rows):,}件"
        )

        print(
            "集計機種数: "
            f"{len(machine_ids):,}機種"
        )

        print(
            "集計日数: "
            f"{len(record_dates):,}日"
        )

        if not history_rows:
            print(
                "保存対象の履歴がありません。"
            )
            return

        (
            inserted_count,
            updated_count,
            skipped_count,
        ) = save_price_history(
            connection,
            history_rows,
        )

        connection.commit()

        total_count = count_history_rows(
            connection
        )

        (
            oldest_date,
            latest_date,
        ) = get_history_date_range(
            connection
        )

        print("=" * 70)

        print(
            "新規保存件数: "
            f"{inserted_count:,}件"
        )

        print(
            "更新件数: "
            f"{updated_count:,}件"
        )

        print(
            "スキップ件数: "
            f"{skipped_count:,}件"
        )

        print(
            "price_history総件数: "
            f"{total_count:,}件"
        )

        print(
            "履歴期間: "
            f"{oldest_date or '-'}"
            " ～ "
            f"{latest_date or '-'}"
        )

        print("=" * 70)

    except Exception:
        connection.rollback()
        raise

    finally:
        connection.close()


# ==================================================
# 実行
# ==================================================

if __name__ == "__main__":
    try:
        backfill_price_history()

        elapsed_time = (
            time.time()
            - start_time
        )

        print("-" * 70)
        print(
            "過去価格履歴の再集計が完了しました。"
        )

        print(
            "処理時間: "
            f"{elapsed_time:.2f}秒"
        )

        print("-" * 70)

    except sqlite3.Error as error:
        print("-" * 70)
        print(
            "SQLite処理でエラーが発生しました。"
        )

        print(
            f"{type(error).__name__}: "
            f"{error}"
        )

        raise

    except Exception as error:
        print("-" * 70)
        print(
            "過去価格履歴の再集計処理で"
            "エラーが発生しました。"
        )

        print(
            f"{type(error).__name__}: "
            f"{error}"
        )

        raise

過去価格履歴の再集計を開始します。
データベース: C:\Users\Owner\myenv310\soubanavi\db\data.db
商品テーブル: result_table
履歴テーブル: price_history
開始日: 指定なし
終了日: 自動
今日を含める: False
既存履歴を更新: True
取得商品数: 20,796件
集計対象外商品数: 0件
集計履歴数: 7,785件
集計機種数: 4,422機種
集計日数: 4日
新規保存件数: 110件
更新件数: 7,675件
スキップ件数: 0件
price_history総件数: 14,066件
履歴期間: 2026-07-19 ～ 2026-07-29
----------------------------------------------------------------------
過去価格履歴の再集計が完了しました。
処理時間: 1.38秒
----------------------------------------------------------------------
